# Project 94 — Local Notebook Refactor Assistant
## Improve a Messy Notebook into a Cleaner Learning Notebook

**Stack:** Ollama · LangChain · Jupyter

## Project Overview

This notebook builds a **local notebook refactor assistant** that analyzes messy
Jupyter notebook cells, detects quality issues (code smells, missing explanations,
poor structure), and generates improved versions — turning rough notebooks into
clean, educational learning notebooks.

Everything runs **locally** via Ollama. No code leaves your machine.

### What You Will Learn

1. How to represent notebook cells as structured data for LLM analysis
2. How to detect **code smells** in notebook code cells
3. How to detect **notebook-level issues** (missing markdown, poor ordering)
4. How to generate **improved code** and **explanatory markdown**
5. How to produce a **before/after comparison** for review
6. Practical patterns for LLM-assisted notebook improvement

## Prerequisites

| Requirement | Details |
|---|---|
| **Ollama** | Running locally with `qwen3:8b` pulled |
| **Python packages** | `langchain`, `langchain-ollama` |

```bash
ollama pull qwen3:8b
```

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Imports and Configuration

In [ ]:
import re
import textwrap

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

OLLAMA_MODEL = "qwen3:8b"
TEMPERATURE = 0.2

print("Configuration ready.")

## Step 2 — Initialize LLM

We use **Qwen3 8B** via Ollama for all refactoring tasks.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

test_msg = llm.invoke("Reply with only: OK")
print(f"LLM ready: {test_msg.content.strip()[:20]}")

## Step 3 — Define a Sample Messy Notebook

We represent a messy notebook as a list of cells, each with a `type`
(code or markdown) and `source`. This simulates a real `.ipynb`
structure without needing to parse JSON.

This sample notebook has common problems:
- Giant code cells with no explanation
- Poor variable names
- Missing markdown between code cells
- No title or structure
- Hardcoded values
- No imports organized at the top

In [ ]:
MESSY_NOTEBOOK = [
    {"type": "code", "source": textwrap.dedent("""\
        import pandas as pd
        import numpy as np
        from sklearn.model_selection import train_test_split
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import accuracy_score
        import matplotlib.pyplot as plt
        d = pd.read_csv('data.csv')
        print(d.shape)
        d.head()
    """)},
    {"type": "code", "source": textwrap.dedent("""\
        d = d.dropna()
        d = d[d['age'] > 0]
        d = d[d['age'] < 150]
        d['name'] = d['name'].str.strip().str.title()
        d['bmi'] = d['weight'] / (d['height']/100)**2
        d['age_group'] = pd.cut(d['age'], bins=[0,18,35,50,65,150], labels=['teen','young','mid','senior','elderly'])
        d['high_risk'] = ((d['bmi'] > 30) & (d['age'] > 50)).astype(int)
        print(d.shape)
        print(d['high_risk'].value_counts())
    """)},
    {"type": "code", "source": textwrap.dedent("""\
        plt.figure(figsize=(10,6))
        plt.hist(d['age'], bins=30)
        plt.title('age')
        plt.show()
        plt.figure(figsize=(10,6))
        plt.hist(d['bmi'], bins=30)
        plt.title('bmi')
        plt.show()
        d.corr().style.background_gradient()
    """)},
    {"type": "code", "source": textwrap.dedent("""\
        X = d.drop(['high_risk','name'], axis=1)
        X = pd.get_dummies(X)
        y = d['high_risk']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
        m = RandomForestClassifier(n_estimators=100)
        m.fit(X_train, y_train)
        p = m.predict(X_test)
        print(accuracy_score(y_test, p))
    """)},
]

print(f"Messy notebook has {len(MESSY_NOTEBOOK)} cells:")
for i, cell in enumerate(MESSY_NOTEBOOK):
    lines = cell['source'].strip().count('\n') + 1
    print(f"  Cell {i}: {cell['type']} ({lines} lines)")

## Step 4 — Analyze Individual Code Cells

We analyze each code cell for **code smells** — poor naming, missing
comments, hardcoded values, overly long cells, etc.

In [ ]:
CELL_ANALYSIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a notebook quality reviewer. Analyze this code cell
from a Jupyter notebook and identify quality issues.

For each issue found, state:
- ISSUE: short name (e.g., poor_naming, no_comments, hardcoded_path)
- SEVERITY: critical / high / medium / low
- DESCRIPTION: what is wrong and why it matters
- FIX: how to improve it

Also provide:
- QUALITY SCORE: 1-10 (10 = excellent)
- CELL PURPOSE: one sentence describing what this cell does

Be specific — reference variable names and line patterns."""),
    ("human", "Cell {cell_num} (code):\n```python\n{code}\n```")
])

cell_analysis_chain = CELL_ANALYSIS_PROMPT | llm | StrOutputParser()

cell_analyses = []
for i, cell in enumerate(MESSY_NOTEBOOK):
    analysis = cell_analysis_chain.invoke({
        "cell_num": i,
        "code": cell["source"].strip(),
    })
    cell_analyses.append(analysis)
    print(f"\nCELL {i} ANALYSIS")
    print("-" * 50)
    print(analysis[:500])
    if len(analysis) > 500:
        print(f"... ({len(analysis)} chars)")
    print()

## Step 5 — Detect Notebook-Level Issues

Beyond individual cells, a notebook has **structural** issues:
- No title or project description
- No markdown cells between code cells
- Missing section headers
- No EDA explanation
- No conclusion or summary

We send the full notebook outline to the LLM for structural analysis.

In [ ]:
NOTEBOOK_ANALYSIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a notebook structure reviewer. Analyze the overall
structure of this Jupyter notebook and identify structural issues.

A good learning notebook should have:
- A title and project description at the top
- A markdown cell BEFORE each code cell explaining what it does and why
- Clear section headers (## Section Name)
- Logical ordering: imports → data loading → EDA → preprocessing → modeling → evaluation
- A summary or conclusion at the end
- Reproducibility notes (random seeds, data source)

List specific structural issues and suggest a better section outline."""),
    ("human", "Notebook structure:\n\n{outline}")
])

# Build outline
outline_parts = []
for i, cell in enumerate(MESSY_NOTEBOOK):
    first_lines = cell["source"].strip().split("\n")[:3]
    preview = " | ".join(line.strip() for line in first_lines)
    outline_parts.append(f"Cell {i} [{cell['type']}]: {preview}")

outline_text = "\n".join(outline_parts)

notebook_analysis_chain = NOTEBOOK_ANALYSIS_PROMPT | llm | StrOutputParser()
structure_analysis = notebook_analysis_chain.invoke({"outline": outline_text})

print("NOTEBOOK STRUCTURE ANALYSIS")
print("=" * 60)
print(structure_analysis)

## Step 6 — Generate Improved Code Cells

For each messy code cell, we ask the LLM to produce a **refactored version**
with better naming, comments, structure, and practices.

In [ ]:
REFACTOR_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Refactor this Jupyter notebook code cell to be cleaner and
more educational. Apply these improvements:

- Use descriptive variable names (not single letters)
- Add inline comments for non-obvious logic
- Use constants instead of magic numbers
- Break long lines
- Add type hints where helpful
- Use descriptive plot titles and labels
- Set random seeds for reproducibility

Return ONLY the improved Python code — no markdown fences, no explanation."""),
    ("human", "Original cell:\n{code}\n\nIssues found:\n{issues}")
])

def strip_fences(text: str) -> str:
    """Remove ```python ... ``` wrappers if present."""
    text = text.strip()
    if text.startswith("```"):
        first_nl = text.index("\n")
        text = text[first_nl + 1:]
    if text.endswith("```"):
        text = text[:-3].rstrip()
    return text

refactor_chain = REFACTOR_PROMPT | llm | StrOutputParser()

refactored_cells = []
for i, cell in enumerate(MESSY_NOTEBOOK):
    improved_raw = refactor_chain.invoke({
        "code": cell["source"].strip(),
        "issues": cell_analyses[i][:500],
    })
    improved = strip_fences(improved_raw)
    refactored_cells.append(improved)

    print(f"\nREFACTORED CELL {i}")
    print("=" * 50)
    print(improved[:600])
    if len(improved) > 600:
        print(f"... ({len(improved)} chars)")
    print()

## Step 7 — Generate Markdown Explanations

A good learning notebook has a **markdown cell before each code cell**
explaining what the code does and why. We generate these for each
refactored cell.

In [ ]:
MARKDOWN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Write a markdown cell to place BEFORE this code cell in a
Jupyter notebook. The markdown should:

1. Have a section header (## Section Name)
2. Explain WHAT the code does in 1-2 sentences
3. Explain WHY this step matters in the workflow
4. Mention any key decisions (e.g., why a threshold was chosen)

Keep it concise — 3-6 lines of markdown. Use a teaching tone.
Return ONLY the markdown text — no code fences."""),
    ("human", "Code cell:\n{code}")
])

markdown_chain = MARKDOWN_PROMPT | llm | StrOutputParser()

markdown_cells = []
for i, code in enumerate(refactored_cells):
    md = markdown_chain.invoke({"code": code})
    md = strip_fences(md)
    markdown_cells.append(md)
    print(f"MARKDOWN FOR CELL {i}")
    print("-" * 40)
    print(md)
    print()

## Step 8 — Generate Title and Summary Cells

Every good notebook needs a **title cell** at the top and a **summary cell**
at the bottom. We generate both.

In [ ]:
TITLE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Based on the code cells below, write a notebook title cell in
markdown. Include:
1. A title (# Title)
2. A one-paragraph project description
3. A bullet list of what the notebook covers

Return ONLY markdown text."""),
    ("human", "Notebook code summary:\n{summary}")
])

SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Write a summary / conclusion cell for the end of a Jupyter
notebook. Include:
1. ## Summary header
2. Key findings or results
3. What could be improved next

Keep it under 10 lines of markdown. Return ONLY markdown text."""),
    ("human", "Notebook code summary:\n{summary}")
])

# Build a summary of what the notebook does
code_summary = "\n---\n".join(c[:200] for c in refactored_cells)

title_chain = TITLE_PROMPT | llm | StrOutputParser()
summary_chain = SUMMARY_PROMPT | llm | StrOutputParser()

title_cell = strip_fences(title_chain.invoke({"summary": code_summary}))
summary_cell = strip_fences(summary_chain.invoke({"summary": code_summary}))

print("GENERATED TITLE CELL")
print("=" * 50)
print(title_cell)
print("\nGENERATED SUMMARY CELL")
print("=" * 50)
print(summary_cell)

## Step 9 — Assemble the Refactored Notebook

We combine all generated pieces into the final clean notebook structure:

```
Title cell (markdown)
For each original cell:
  → Explanation cell (markdown)
  → Refactored code cell
Summary cell (markdown)
```

In [ ]:
clean_notebook = []

# Title
clean_notebook.append({"type": "markdown", "source": title_cell})

# Interleaved markdown + code
for md, code in zip(markdown_cells, refactored_cells):
    clean_notebook.append({"type": "markdown", "source": md})
    clean_notebook.append({"type": "code", "source": code})

# Summary
clean_notebook.append({"type": "markdown", "source": summary_cell})

print("REFACTORED NOTEBOOK STRUCTURE")
print("=" * 50)
print(f"Original: {len(MESSY_NOTEBOOK)} cells (all code, no markdown)")
print(f"Refactored: {len(clean_notebook)} cells")
print()

md_count = sum(1 for c in clean_notebook if c['type'] == 'markdown')
code_count = sum(1 for c in clean_notebook if c['type'] == 'code')
print(f"  Markdown cells: {md_count}")
print(f"  Code cells:     {code_count}")
print()

for i, cell in enumerate(clean_notebook):
    first_line = cell['source'].split('\n')[0].strip()[:70]
    print(f"  [{cell['type']:>8}] {first_line}")

## Step 10 — Before/After Comparison

We ask the LLM to provide a structured comparison between the original
messy notebook and the refactored version.

In [ ]:
COMPARE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Compare the BEFORE and AFTER versions of a refactored notebook.
Provide a structured comparison covering:

1. STRUCTURE: Cell count, markdown/code ratio, section organization
2. READABILITY: Variable naming, comments, explanation quality
3. BEST PRACTICES: Random seeds, type hints, constants vs magic numbers
4. EDUCATIONAL VALUE: Could a learner follow the notebook top-to-bottom?
5. OVERALL IMPROVEMENT SCORE: 1-10

Be concise — 1-2 sentences per dimension."""),
    ("human", "BEFORE ({before_count} cells, all code):\n{before}\n\n"
     "AFTER ({after_count} cells, {md_count} markdown + {code_count} code):\n{after}")
])

compare_chain = COMPARE_PROMPT | llm | StrOutputParser()

before_preview = "\n---\n".join(c["source"].strip()[:150] for c in MESSY_NOTEBOOK)
after_preview = "\n---\n".join(c["source"][:150] for c in clean_notebook)

comparison = compare_chain.invoke({
    "before_count": len(MESSY_NOTEBOOK),
    "before": before_preview,
    "after_count": len(clean_notebook),
    "md_count": md_count,
    "code_count": code_count,
    "after": after_preview,
})

print("BEFORE / AFTER COMPARISON")
print("=" * 60)
print(comparison)

## Step 11 — Validate Refactored Code

We verify that all refactored code cells at least **compile** without
syntax errors. This catches cases where the LLM generated broken code.

In [ ]:
print("CODE VALIDATION")
print("=" * 50)

all_ok = True
for i, code in enumerate(refactored_cells):
    try:
        compile(code, f"refactored_cell_{i}", "exec")
        print(f"  Cell {i}: ✓ syntax OK")
    except SyntaxError as e:
        print(f"  Cell {i}: ✗ SyntaxError — {e}")
        all_ok = False

if all_ok:
    print("\nAll refactored cells compile successfully.")
else:
    print("\nSome cells have syntax errors — LLM output may need manual review.")

## Evaluation Summary

| Dimension | How We Evaluated |
|---|---|
| **Cell-level analysis** | LLM detected code smells per cell (Step 4) |
| **Notebook structure** | LLM identified missing markdown, ordering issues (Step 5) |
| **Code quality** | Refactored cells checked for syntax validity (Step 11) |
| **Markdown quality** | Generated explanations reviewed for clarity (Step 7) |
| **Overall improvement** | Before/after comparison scored by LLM (Step 10) |
| **Completeness** | Final notebook has title, explanations, and summary |

### Known Limitations

- **No execution validation**: We check syntax but cannot run the code
  without the actual dataset.
- **Context loss**: The LLM refactors cells independently — it may not
  preserve variable names across cells consistently.
- **Markdown quality varies**: Generated explanations may be generic.
  Domain-specific notebooks need human review.
- **Large notebooks**: Very long notebooks should be processed in chunks
  to stay within context limits.
- **Style preferences**: "Clean" is subjective — the refactored style
  reflects general best practices, not team-specific conventions.

## How to Improve This Project

1. **File-based workflow** — read real `.ipynb` files, refactor, and write back
2. **Cross-cell context** — pass all cells together so the LLM preserves
   variable names and data flow across cells
3. **Style configuration** — let users specify naming conventions, section
   structure, and explanation depth preferences
4. **Diff view** — show a side-by-side diff of original vs. refactored code
5. **Iterative refinement** — re-analyze refactored cells and fix remaining issues
6. **Notebook scoring rubric** — define a quantitative rubric and track
   improvement across multiple refactoring passes

## What You Learned

- **Cell-level code analysis** — detecting smells in individual notebook cells
- **Notebook-level structure analysis** — identifying missing sections and ordering issues
- **Code refactoring** — generating improved code with better naming and practices
- **Markdown generation** — creating explanatory cells for educational notebooks
- **Notebook assembly** — combining title, explanations, code, and summary into a clean structure
- **Before/after comparison** — evaluating improvement quality quantitatively